# 05 — File Operations

**Yang akan lo pelajari:**
- Upload file ke Jupyter (lokal & Colab)
- Baca file teks, CSV, JSON, Excel
- Tulis dan simpan file
- Download hasil output
- Navigasi direktori

---

## 1. Upload File

In [ ]:
# Deteksi environment: Colab atau Jupyter biasa
import sys

IS_COLAB = 'google.colab' in sys.modules
print(f"Environment: {'Google Colab' if IS_COLAB else 'Jupyter Lokal'}")

In [ ]:
# CARA 1: Upload di Google Colab
if IS_COLAB:
    from google.colab import files
    uploaded = files.upload()  # muncul dialog upload
    
    # uploaded = dict {filename: bytes}
    for filename, content in uploaded.items():
        print(f"Uploaded: {filename} ({len(content)} bytes)")
else:
    print("Bukan Colab — pakai cara drag & drop atau path langsung")

In [ ]:
# CARA 2: Upload via ipywidgets (Jupyter lokal)
try:
    import ipywidgets as widgets
    from IPython.display import display
    
    uploader = widgets.FileUpload(
        accept='',       # kosong = terima semua
        multiple=False   # satu file saja
    )
    
    display(uploader)
    print("Klik tombol di atas untuk upload file")
    
    # Setelah upload, akses dengan:
    # content = uploader.data[0]  # bytes
    # filename = uploader.metadata[0]['name']
except ImportError:
    print("ipywidgets tidak terinstall: pip install ipywidgets")

## 2. Buat File Contoh untuk Latihan

In [ ]:
import os
import json
import csv
from pathlib import Path

# Buat direktori untuk latihan
DATA_DIR = Path('sample_data')
DATA_DIR.mkdir(exist_ok=True)

# 1. File teks biasa
with open(DATA_DIR / 'catatan.txt', 'w') as f:
    f.write("Baris pertama\n")
    f.write("Baris kedua\n")
    f.write("Baris ketiga\n")

# 2. File CSV
csv_data = [
    ['nama', 'umur', 'kota', 'nilai'],
    ['Budi', 25, 'Jakarta', 85],
    ['Ani', 23, 'Bandung', 92],
    ['Ciko', 27, 'Surabaya', 78],
    ['Dewi', 24, 'Medan', 95],
    ['Eko', 26, 'Yogyakarta', 88],
]

with open(DATA_DIR / 'data.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)

# 3. File JSON
json_data = {
    "project": "AI Notebooks",
    "version": "1.0",
    "topics": ["numpy", "pandas", "matplotlib"],
    "stats": {"notebooks": 11, "readme_files": 11}
}

with open(DATA_DIR / 'config.json', 'w') as f:
    json.dump(json_data, f, indent=2)

print("Sample files berhasil dibuat:")
for f in DATA_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size} bytes)")

## 3. Baca File Teks

In [ ]:
# Baca seluruh isi file sekaligus
with open(DATA_DIR / 'catatan.txt', 'r') as f:
    isi = f.read()

print(isi)
print(type(isi))    # str

In [ ]:
# Baca per baris
with open(DATA_DIR / 'catatan.txt', 'r') as f:
    baris = f.readlines()   # list of strings

print(baris)

# Bersihkan newline
baris_bersih = [b.strip() for b in baris]
print(baris_bersih)

In [ ]:
# Cara lebih efisien untuk file besar — iterasi langsung
print("Membaca line by line:")
with open(DATA_DIR / 'catatan.txt', 'r') as f:
    for i, line in enumerate(f, 1):
        print(f"  Line {i}: {line.strip()}")

## 4. Baca dan Tulis CSV

In [ ]:
import pandas as pd

# Cara paling umum: pakai Pandas
df = pd.read_csv(DATA_DIR / 'data.csv')
print(df)
print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# read_csv dengan berbagai opsi
df2 = pd.read_csv(
    DATA_DIR / 'data.csv',
    # sep=';',          # separator berbeda (default: ',')
    # header=None,      # tidak ada header
    # index_col='nama', # set kolom sebagai index
    # nrows=3,          # baca 3 baris saja
    # skiprows=[1, 2],  # skip baris tertentu
    # encoding='utf-8', # encoding file
)
print(df2.dtypes)    # tipe data tiap kolom

In [ ]:
# Simpan DataFrame ke CSV
df_modified = df.copy()
df_modified['nilai_x2'] = df_modified['nilai'] * 2

# to_csv — index=False supaya tidak ada kolom nomor
df_modified.to_csv(DATA_DIR / 'data_modified.csv', index=False)
print("Berhasil disimpan ke data_modified.csv")

# Verifikasi
pd.read_csv(DATA_DIR / 'data_modified.csv')

## 5. Baca dan Tulis JSON

In [ ]:
import json

# Baca JSON
with open(DATA_DIR / 'config.json', 'r') as f:
    config = json.load(f)

print(type(config))          # dict
print(config['project'])     # AI Notebooks
print(config['topics'])      # list

# Pretty print
print(json.dumps(config, indent=2))

In [ ]:
# Tulis JSON
new_data = {
    "hasil": [85, 92, 78, 95, 88],
    "rata_rata": 87.6,
    "timestamp": "2025-01-01"
}

with open(DATA_DIR / 'hasil.json', 'w') as f:
    json.dump(new_data, f, indent=2, ensure_ascii=False)

print("JSON berhasil ditulis")

## 6. Baca Excel

In [ ]:
# Buat file Excel sederhana dulu
try:
    import openpyxl
    
    df_excel = pd.DataFrame({
        'Produk': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'],
        'Harga': [15000000, 350000, 500000, 3500000],
        'Stok': [10, 50, 30, 15]
    })
    
    df_excel.to_excel(DATA_DIR / 'produk.xlsx', index=False, sheet_name='Inventaris')
    print("Excel berhasil dibuat")
    
    # Baca Excel
    df_read = pd.read_excel(DATA_DIR / 'produk.xlsx', sheet_name='Inventaris')
    print(df_read)
    
except ImportError:
    print("openpyxl tidak ada: pip install openpyxl")

## 7. Download File dari Notebook

In [ ]:
# CARA 1: Google Colab — download langsung
if IS_COLAB:
    from google.colab import files
    files.download('sample_data/data_modified.csv')
else:
    print("Di Jupyter lokal: file sudah ada di direktori, tinggal ambil langsung")
    print(f"Path: {Path('sample_data/data_modified.csv').resolve()}")

In [ ]:
# CARA 2: Buat link download via HTML (Jupyter lokal)
from IPython.display import FileLink, display

display(FileLink('sample_data/data_modified.csv'))
print("↑ Klik link di atas untuk download")

## 8. Navigasi dan Manajemen File

In [ ]:
from pathlib import Path
import shutil

# List semua file di direktori
print("Files di sample_data/:")
for f in sorted(DATA_DIR.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:<25} {size:>6} bytes")

In [ ]:
# Cari file dengan pattern
csv_files = list(DATA_DIR.glob('*.csv'))
print(f"File CSV: {[f.name for f in csv_files]}")

# Cari rekursif
all_files = list(Path('.').rglob('*.json'))
print(f"Semua JSON: {[str(f) for f in all_files]}")

In [ ]:
# Copy, move, rename
src = DATA_DIR / 'catatan.txt'
dst = DATA_DIR / 'catatan_backup.txt'

shutil.copy(src, dst)
print(f"Copied: {dst.name}")

# Rename
dst.rename(DATA_DIR / 'backup.txt')
print("Renamed to backup.txt")

# Hapus file
(DATA_DIR / 'backup.txt').unlink()
print("Deleted backup.txt")

In [ ]:
# Cleanup — hapus direktori latihan
shutil.rmtree(DATA_DIR)
print(f"Direktori {DATA_DIR} dihapus")

---
## ➡️ Next
Lanjut ke **[06_numpy_fundamentals.ipynb](06_numpy_fundamentals.ipynb)** — array multidimensi, operasi matematika cepat, dan dasar semua ML numerik.